In [13]:
from datetime import datetime, timezone
import pandas as pd

In [14]:
df = pd.read_csv("fills.csv")
df.columns

Index(['Unnamed: 0', 'account', 'accountId', 'aggressorIndicator', 'algoId',
       'avgPx', 'brokerId', 'cumQty', 'currUserId', 'deltaQty',
       'exchLeavesQty', 'exchOrderAssoc', 'execId', 'execInst', 'execType',
       'externallyCreated', 'instrumentId', 'lastPx', 'lastQty',
       'manualOrderIndicator', 'marketId', 'messageType',
       'multiLegReportingType', 'ordStatus', 'ordType', 'orderId',
       'parentOrderId', 'parties', 'recordId', 'secondaryClOrdId',
       'secondaryExecId', 'secondaryOrderId', 'securityDesc',
       'senderLocationId', 'senderSubId', 'side', 'source', 'syntheticType',
       'timeInForce', 'timeSentClient', 'timeSentTT', 'timeStamp', 'tradeDate',
       'transactTime', 'uniqueExecId', 'lastLiquidityInd', 'positionEffect',
       'tradingVenueTradeId', 'tradeMatchId', 'parentInstrumentId',
       'fillsGroup', 'allocId', 'orderCrossPreventionType'],
      dtype='str')

In [15]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo  

def convert_ns_to_time(ns_timestamp):
    if ns_timestamp is None:
        return None

    # Convert nanoseconds to seconds (float keeps precision)
    seconds = ns_timestamp / 1_000_000_000

    # Create UTC datetime
    dt_utc = datetime.fromtimestamp(seconds, tz=timezone.utc)

    # Convert to IST
    dt_ist = dt_utc.astimezone(ZoneInfo("Asia/Kolkata"))

    # Return milliseconds precision
    return dt_ist.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]


In [16]:
df['utc_time'] = df['transactTime'].apply(convert_ns_to_time)

In [17]:
df.to_csv("fills_2.csv")

# FIFO PNL from tt-export.csv

Reads a TT fills export (no header row) and computes realized PNL per contract using strict FIFO lot matching (oldest lot closed first) — same methodology as the app's `/api/pnl/overview` backend.

Prices in this export are already in **display** terms (what you see in the TT UI), so `tick_size`/`tick_value` below must also be the **display**-unit tick spec for each contract (e.g. CME GC: tick_size=0.1, tick_value=$10), not TT's raw/internal tick spec.

**Edit the parameters cell below** — set the CSV path and each contract's tick_size/tick_value — before running.

In [9]:
CSV_PATH = r"C:\Users\Harshvardan\Desktop\Workspace\pnl dashboard TT\tt-export.csv"

TICK_SPECS = {
    "GDU Dec26":                  {"tick_size": 0.01, "tick_value": 0.01},
    "GDU Dec26-Feb27 Calendar":   {"tick_size": 0.01, "tick_value": 0.01},
    "GC Dec26":                   {"tick_size": 0.1,  "tick_value": 10.0},
    "GC Dec26-Feb27 Calendar":    {"tick_size": 0.1,  "tick_value": 10.0},
}


In [10]:
import pandas as pd
from datetime import datetime
from collections import deque

COLUMNS = [
    "date", "time", "exchange", "contract", "side", "qty", "price",
    "order_type", "routing", "account", "trader", "trader2",
    "exec_id", "order_id", "_blank",
]

fills = pd.read_csv(CSV_PATH, header=None, names=COLUMNS)


fills["ts"] = pd.to_datetime(
    fills["date"] + " " + fills["time"], format="%d%b%y %H:%M:%S.%f"
)
fills = fills.sort_values("ts").reset_index(drop=True)

missing_specs = sorted(set(fills["contract"]) - set(TICK_SPECS))
if missing_specs:
    raise ValueError(f"No tick_size/tick_value configured for: {missing_specs} — add them to TICK_SPECS above.")

fills[["date", "time", "contract", "side", "qty", "price"]].head()

,date,time,contract,side,qty,price
0,02Sep26,06:22:27.319,GDU Dec26-Feb27 Calendar,B,10,1.15
1,02Sep26,06:38:38.075,GDU Dec26,S,3,140.15
2,02Sep26,06:38:38.503,GC Dec26,B,1,4358.30
3,02Sep26,06:40:49.588,GDU Dec26,S,3,140.22
4,02Sep26,06:40:49.848,GC Dec26,B,1,4360.90


In [11]:
def fifo_pnl(contract_fills: pd.DataFrame, tick_size: float, tick_value: float) -> dict:
  
    buy_qty = contract_fills.loc[contract_fills["side"] == "B", "qty"].sum()
    sell_qty = contract_fills.loc[contract_fills["side"] == "S", "qty"].sum()

    open_lots = deque()  # [[qty, price], ...] — all same side while non-empty
    open_side = 0  # 1 = long, -1 = short, 0 = flat
    realized = 0.0

    qty_epsilon = max(1e-9, (buy_qty + sell_qty) * 1e-9)

    for _, f in contract_fills.iterrows():
        side = 1 if f["side"] == "B" else -1
        qty = f["qty"]
        price = f["price"]

        if open_side == 0 or side == open_side:
            open_lots.append([qty, price])
            open_side = side
            continue

        remaining = qty
        while remaining > qty_epsilon and open_lots:
            lot = open_lots[0]
            matched = min(remaining, lot[0])
            price_diff = (price - lot[1]) if open_side == 1 else (lot[1] - price)
            realized += (price_diff / tick_size) * tick_value * matched
            lot[0] -= matched
            remaining -= matched
            if lot[0] <= qty_epsilon:
                open_lots.popleft()

        if remaining > qty_epsilon:
            open_side = side
            open_lots.append([remaining, price])
        elif not open_lots:
            open_side = 0

    if open_lots:
        open_qty = sum(lot[0] for lot in open_lots)
        avg_open_price = sum(lot[0] * lot[1] for lot in open_lots) / open_qty
        open_qty *= open_side
    else:
        open_qty = 0.0
        avg_open_price = 0.0

    return {
        "buy_qty": buy_qty,
        "sell_qty": sell_qty,
        "open_qty": open_qty,
        "avg_open_price": avg_open_price,
        "realized_pnl": realized,
    }

In [12]:
results = []
for contract, group in fills.groupby("contract"):
    spec = TICK_SPECS[contract]
    r = fifo_pnl(group, spec["tick_size"], spec["tick_value"])
    results.append({
        "contract": contract,
        "buy_qty": r["buy_qty"],
        "sell_qty": r["sell_qty"],
        "open_qty": round(r["open_qty"], 4),
        "avg_open_price": round(r["avg_open_price"], 4),
        "realized_pnl": round(r["realized_pnl"], 2),
    })

pnl_df = pd.DataFrame(results).sort_values("contract").reset_index(drop=True)

total_row = {
    "contract": "TOTAL",
    "buy_qty": pnl_df["buy_qty"].sum(),
    "sell_qty": pnl_df["sell_qty"].sum(),
    "open_qty": round(pnl_df["open_qty"].sum(), 4),
    "avg_open_price": None,  # not meaningful summed across different contracts
    "realized_pnl": round(pnl_df["realized_pnl"].sum(), 2),
}

pnl_df = pd.concat([pnl_df, pd.DataFrame([total_row])], ignore_index=True)
pnl_df

,contract,buy_qty,sell_qty,open_qty,avg_open_price,realized_pnl
0,GC Dec26,8,5,3.0,4350.8667,-4470.00
1,GC Dec26-Feb27 Calendar,2,2,0.0,0.0,40.00
2,GDU Dec26,185,194,-9.0,141.68,-15.64
3,GDU Dec26-Feb27 Calendar,52,42,10.0,1.14,-0.10
4,TOTAL,247,243,4.0,None,-4445.74


In [ ]:
whatever you have done in table UIdont is too bad use the ui ux skill.
firsly the double click should only happen when clicked on column border
- then it should warp content and no gap.
- and one double click should make all columns content wrapped
- hovering on columns border should show changed cursor to that icon of wrapping